In [28]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [29]:
df = pd.read_csv('/content/insurance.csv')

In [30]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
25,59,60.2,1.55,30.000000,False,Mysore,government_job,Low
91,38,119.8,1.76,28.467885,False,Bangalore,government_job,Low
26,33,79.0,1.61,23.610000,False,Jaipur,freelancer,Medium
35,59,59.3,1.69,43.280000,True,Chandigarh,private_job,Medium
92,37,62.7,1.85,30.000000,True,Lucknow,government_job,Low


In [31]:
df_feat = df.copy()

## Feature Engineering

In [32]:
# Feature 1: BMI
df_feat['bmi'] = df_feat['weight'] / (df_feat['height'] ** 2)

In [33]:
# Feature 2: Age group
def age_group(age):
  if age < 25:
    return 'young'
  elif age < 45:
    return 'adult'
  elif age < 60:
    return 'middle_aged'
  else:
    return 'senior'

In [34]:
df_feat['age_group'] = df_feat['age'].apply(age_group)

In [35]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
  if row['smoker'] and row['bmi'] > 30:
    return 'high'
  elif row['smoker'] or row['bmi'] > 27:
    return 'medium'
  else:
    return 'low'

In [36]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis=1)

In [37]:
# Feature 4: City Tier

tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [38]:
df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [39]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
92,30.00,government_job,18.319942,adult,medium,2,Low
80,50.00,unemployed,34.350461,middle_aged,medium,2,High
9,43.07,business_owner,24.858833,middle_aged,low,1,Low
46,25.57,unemployed,33.672766,adult,high,1,High
43,1.56,retired,29.308163,senior,medium,1,Medium


In [40]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [41]:
X.head()

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,medium,2,2.92,retired
1,30.189017,adult,medium,1,34.28,freelancer
2,21.118382,adult,low,2,36.64,freelancer
3,45.535900,young,high,1,3.34,student
4,24.296875,senior,medium,2,3.94,retired


In [42]:
y.head()

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High


In [43]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [44]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

## Model Training

In [45]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [46]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [47]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.9

In [48]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)